In [1]:
# 1. Import libraries and helper functions

import pandas as pd
from pathlib import Path
import os

def normalize_text(series):
    return series.astype(str).str.strip().str.title()

In [2]:
# 2. Path configuration

BASE_DIR = Path(os.environ.get("MASKING_DATA_PATH", "../data"))
RAW_DIR = BASE_DIR / "raw" / "us"
PROCESSED_DIR = BASE_DIR / "processed" / "us"

In [3]:
# 3. Load dataset

df = pd.read_excel(RAW_DIR / "sales_raw.xlsx").dropna(how='all')

In [4]:
# 4. Remap ITEM_IDs removed during product deduplication

# During products cleaning, 648 duplicate ITEM_IDs were identified (same
# product registered more than once) and removed from the dimension,
# keeping only the canonical (oldest) ITEM_ID per product. Any sales line
# referencing a removed ITEM_ID must be reassigned to its canonical ID -
# otherwise it would become an orphan record with no matching product.

item_id_map_df = pd.read_csv(PROCESSED_DIR / "item_id_map.csv")
item_id_map = dict(zip(item_id_map_df['old_item_id'], item_id_map_df['canonical_item_id']))

ids_before = df['ITEM_ID'].copy()
df['ITEM_ID'] = df['ITEM_ID'].replace(item_id_map)
affected = (ids_before != df['ITEM_ID']).sum()

print(f"{affected} sales rows remapped to canonical item_id")

19188 sales rows remapped to canonical item_id


In [5]:
# 5. Validate referential integrity against the product dimension

products_dim = pd.read_csv(PROCESSED_DIR / "dim_products.csv")

orphans = df[~df['ITEM_ID'].astype(str).isin(products_dim['item_id'].astype(str))]
assert len(orphans) == 0, f"{len(orphans)} sales rows with item_id not found in products_dim!"

print(f"{len(orphans)} orphan rows - all ITEM_IDs now match the product dimension")

0 orphan rows - all ITEM_IDs now match the product dimension


In [6]:
# 6. Filter dataset columns

filtered_columns = [
    'ORDER_ID', 'CUSTOMER_ID', 'ITEM_ID', 'SALES_REP', 'INVOICE_DATE', 'INVOICE_TYPE', 'TAX_CODE', 'INVOICE_STATUS', 
    'CUSTOMER_STATE', 'SHIPPED_QTY', 'NET_UNIT_PRICE', 'TOTAL_NET_VALUE' 
]

df = df[filtered_columns].copy()

In [7]:
# 7. Data quality check

def data_quality_check(df, name="dataset", id_cols=None):
    """
    Runs a standard data quality check on any DataFrame.
    id_cols: str, list of str, or None. Pass the column(s) that should be
    unique keys - the function checks each one independently.
    """

    # 1. Header
    print(f"{'='*60}")
    print(f"DATA QUALITY CHECK: {name}")
    print(f"{'='*60}\n")

    # 2. Shape
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

    # 3. Data types
    print("--- Data types ---")
    print(df.dtypes)
    print()

    # 4. Nulls
    print("--- Null values ---")
    nulls = df.isna().sum()
    print(nulls[nulls > 0] if nulls.sum() > 0 else "No nulls found")
    print()

    # 5. Fully duplicated rows (entire row identical)
    print("--- Duplicated rows (entire row) ---")
    print(df.duplicated().sum())
    print()

    # 6. Key column(s) duplicated on their own, regardless of the rest of the row
    if id_cols:
        if isinstance(id_cols, str):
            id_cols = [id_cols]
        for col in id_cols:
            if col not in df.columns:
                print(f"WARNING: id_col '{col}' not found in this dataset - skipping\n")
                continue
            print(f"--- Duplicated '{col}' (key column) ---")
            id_dupes = df[col].duplicated().sum()
            print(id_dupes)
            if id_dupes > 0:
                print(f"WARNING: {id_dupes} duplicate '{col}' found - "
                      f"inspect whether other columns diverge between them:")
                print(df[df[col].duplicated(keep=False)].sort_values(col).head(20))
            print()

    # 7. Categorical columns: unique values + top values (spot inconsistent text)
    print("--- Categorical columns: unique value counts ---")
    for col in df.select_dtypes(include='object').columns:
        print(f"\n{col}: {df[col].nunique()} unique values")
        print(df[col].value_counts().head(10))

    # 8. Numeric columns: descriptive stats (spot outliers, negatives, zeros)
    print("\n--- Numeric columns: descriptive stats ---")
    print(df.describe())

    print(f"\n{'='*60}\n")
    
data_quality_check(df, name="sales_raw", id_cols=None)

DATA QUALITY CHECK: sales_raw

Shape: 340999 rows x 12 columns

--- Data types ---
ORDER_ID                  float64
CUSTOMER_ID                 int64
ITEM_ID                     int64
SALES_REP                  object
INVOICE_DATE       datetime64[ns]
INVOICE_TYPE               object
TAX_CODE                   object
INVOICE_STATUS             object
CUSTOMER_STATE             object
SHIPPED_QTY                 int64
NET_UNIT_PRICE            float64
TOTAL_NET_VALUE           float64
dtype: object

--- Null values ---
INVOICE_TYPE    11
dtype: int64

--- Duplicated rows (entire row) ---
0

--- Categorical columns: unique value counts ---

SALES_REP: 25 unique values
SALES_REP
Carla Gray              14608
Holly Wood              14541
Angie Henderson         14476
Margaret Hawkins DDS    14400
Brian Ramirez           14208
Dylan Miller            14138
Gina Moore              14093
Allison Hill            14036
Tommy Walter            14021
Ryan Munoz              13950
Name: count, 

In [8]:
# 8. Rename columns

df.columns = df.columns.str.lower()

In [9]:
# 9. Filling null values

# 11 rows had a missing INVOICE_TYPE. Filled as 'U' (Unknown/Unclassified)
# to avoid losing these rows from revenue reporting while flagging them for further investigation.

df['invoice_type'] = df['invoice_type'].fillna('U')  # U = Unknown/Unclassified

In [10]:
# 10. Validate before changing anything

# item_id and customer_id are EXPECTED to repeat in a sales fact table
# (the same product/customer appears across many order lines) - they are
# not unique keys here, unlike in the dimension tables. The real
# uniqueness check for this table is the combination of order_id + item_id
# + invoice_type (a specific line item within a specific order).

has_real_order = ~df['order_id'].astype(str).isin(['0', '0.0'])
composite_dupes = df[has_real_order].duplicated(subset=['order_id', 'item_id', 'invoice_type']).sum()
print(f"{composite_dupes} order_id + item_id + invoice_type combinations duplicated (excluding order_id = 0)")

448 order_id + item_id + invoice_type combinations duplicated (excluding order_id = 0)


In [11]:
# 11. Investigate whether these are real duplicates or legitimate repeats

# Same order_id + item_id + invoice_type can legitimately repeat when the
# same product is billed in separate lines (different price/quantity
# lots). Only rows identical across EVERY column represent an actual
# duplicate entry.

real_dupes = df[has_real_order & df.duplicated(subset=['order_id', 'item_id'], keep=False)].sort_values(['order_id', 'item_id'])
print(real_dupes[['order_id', 'item_id', 'shipped_qty', 'total_net_value', 'invoice_type']].head(5))

      order_id  item_id  shipped_qty  total_net_value invoice_type
781    40820.0     3705           20           120.00            B
782    40820.0     3705           20           106.20            B
3272   40861.0      568            6           815.22            S
3289   40861.0      568            2           258.20            S
6494   40895.0     3815           20          2039.00            B


In [12]:
# 12. Remove fully duplicated rows

# A fully identical order line (same order_id, item_id, quantity, value,
# and invoice_type) should never occur - if found, it indicates a
# duplicate entry from the source system or the data generation process.
# This check is kept as a standing safeguard, even when no duplicates are
# found in a given run.

before = len(df)
df = df.drop_duplicates(keep='first')
after = len(df)

print(f"{before - after} fully duplicated rows removed")

0 fully duplicated rows removed


In [13]:
# 13. Final check

assert df.duplicated().sum() == 0, "Fully duplicated rows still remain!"

In [14]:
# 14. Standardize invoice_status values

df['invoice_status'] = df['invoice_status'].replace({'CC': 'CANCELED'})

In [15]:
# 15. Replace sales_rep name with rep_id from the dimension table

reps_dim = pd.read_csv(PROCESSED_DIR / "dim_reps.csv")

df['sales_rep'] = normalize_text(df['sales_rep'])

df = df.merge(
    reps_dim[['rep_id', 'rep_name']],
    left_on='sales_rep',
    right_on='rep_name',
    how='left'
)

# Validate: every sales_rep name must have matched a rep_id
assert df['rep_id'].isna().sum() == 0, "Some sales_rep names did not match any rep_id!"

df = df.drop(columns=['sales_rep', 'rep_name'])

# Insert rep_id next to others ids columns
df = df[['order_id', 'customer_id', 'rep_id', 'item_id', 'invoice_date',
         'invoice_type', 'invoice_status', 'tax_code', 'customer_state',
         'shipped_qty', 'net_unit_price', 'total_net_value']]

In [16]:
# 14. Convert data types

df['order_id'] = df['order_id'].astype(str).replace(r'\.0$', '', regex=True)
df['customer_id'] = df['customer_id'].astype(str)
df['rep_id'] = df['rep_id'].astype(str)
df['item_id'] = df['item_id'].astype(str)
df['invoice_type'] = df['invoice_type'].astype('category')
df['invoice_status'] = df['invoice_status'].astype('category')
df['tax_code'] = df['tax_code'].astype('category')
df['customer_state'] = df['customer_state'].astype('category')

# Remove trailing ".0" left over from float-to-string conversion
df['tax_code'] = df['tax_code'].str.replace(r'\.0$', '', regex=True)

In [17]:
# 15. Business rules (validation flags)

# Goal: ensure only valid transactions count toward revenue, excluding:
# - Bonus orders (free products)
# - Invalid transaction codes (TAX_CODE outside the standard sales pattern,
#   e.g. codes used for purchases, tax adjustments, or internal transfers
#   instead of an actual customer sale)
# - Canceled invoices

# 15.1 Bonus orders
df['is_bonus_order'] = df['invoice_type'].eq('B')

# 15.2 Valid TAX_CODE (real sale operations only)
valid_tax_codes = ['TX_SLS_5102', 'TX_SLS_6102', 'TX_SLS_6108']
df['is_valid_code'] = df['tax_code'].isin(valid_tax_codes)

# 15.3 Canceled invoices
df['is_canceled'] = df['invoice_status'].eq('CANCELED')

# 15.4 Final valid-revenue flag
df['is_valid_sale'] = (
    (~df['is_bonus_order']) &
    (df['is_valid_code']) &
    (~df['is_canceled'])
)

In [18]:
# 16. Final KPI: Net Revenue

# TOTAL_NET_VALUE is the only trusted revenue figure in this dataset
# (SHIPPED_QTY, NET_UNIT_PRICE, and TOTAL_NET_VALUE are reliable;
# GROSS_UNIT_PRICE, DISCOUNT_RATE, and TOTAL_DISCOUNT_VALUE are not).
# Invalid transactions are zeroed out so they don't impact revenue KPIs.

df['net_revenue'] = df['total_net_value'].where(df['is_valid_sale'], 0)

In [19]:
# 17. Filter dataset columns

desired_columns = [
    'order_id', 'customer_id', 'rep_id', 'item_id', 'invoice_date', 'invoice_type', 'invoice_status', 'tax_code', 
    'customer_state', 'shipped_qty',  'net_unit_price', 'total_net_value', 'is_valid_sale', 'net_revenue'
]
df = df[desired_columns]

In [20]:
# 15. Final checks

assert df.duplicated().sum() == 0, "Fully duplicated rows still remain!"
assert df['item_id'].isin(products_dim['item_id'].astype(str)).all(), "Orphan ITEM_ID found!"
assert df['rep_id'].isna().sum() == 0, "Missing rep_id found!"

In [21]:
# 16. Export cleaned dataset for BI 

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "fact_sales.csv", index=False)

print(f"Success! {len(df)} sales records exported.")

Success! 340999 sales records exported.
